# 第 1 天任务：ATS 简历差距分析器

该笔记本构建了一个小型**人工智能驱动的简历助手**，其作用类似于申请人跟踪系统（ATS）审阅者。

给定职位描述后，它会调用 OpenAI API 并返回可操作的指导，说明候选人应在简历中添加哪些内容以提高匹配分数（目标：**70%+**）。

## 你将学到什么

- 使用 `.env` 安全地加载 API 密钥
- 调用OpenAI响应API
- 构建**系统**和**用户**提示以完成重点任务
- 将辅助函数链接到一个简单的管道中

## 先决条件

1. 此文件夹（或项目根目录）中的“.env”文件，其中包含“OPENAI_API_KEY=your-key-here”
2.安装依赖项：`openai`、`python-dotenv`
3.选择正确的Python内核（你的虚拟环境）

## 如何运行

使用“Shift + Enter”**从上到下**执行单元格。下面的每个部分都解释了以下代码的作用。

In [12]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
import os
from dotenv import load_dotenv
from openai import OpenAI

## 1.环境配置

接下来的单元格从“.env”文件加载 OpenAI API 密钥。

|变量|目的|
|---|---|
| `OPENAI_API_KEY` |验证对 OpenAI API 的请求 |

`load_dotenv(override=True)` 读取 `.env` 文件并设置环境变量。 `override=True` 确保本地值优先于任何现有的系统变量。

> **提示：** 切勿将您的 `.env` 文件提交到 git。将 API 密钥保密。

In [13]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
load_dotenv(override=True)
apiKey = os.getenv("OPENAI_API_KEY")

## 2.OpenAI 客户端

“OpenAI()”客户端自动从环境中读取“OPENAI_API_KEY”——创建客户端时无需手动传递密钥。

In [14]:
client = OpenAI()

## 3. 提示设计和管道功能

下面的单元格定义了完整的请求管道：

```
jobDescription
    → generateInput()       # builds system + user messages
    → getResponseFromAI()   # calls the LLM
    → getJobDescriptionKeyPoints()  # top-level entry point
```

### 功能概述

|功能|角色 |
|---|---|
| `callLLM（输入）` |通过响应 API 向“gpt-4.1-mini”发送请求 |
| `generateUserInput(jobDescription)` |将职位描述包含在用户提示中 |
| `系统提示` |指示模型充当 ATS 审核者 |
| `generateInput(jobDescription)` |将系统+用户消息组合成API输入格式 |
| `getResponseFromAI（输入）` |调用 LLM 并提取文本响应 |
| `getJobDescriptionKeyPoints(jobDescription)` |协调整个流程|

系统提示告诉模型：
- 从职位描述中确定所需技能
- 建议在简历中添加哪些内容以通过 70% ATS 阈值
- 在 Markdown 中简洁回复

In [15]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def callLLM(input): 
    return client.responses.create(
        model="gpt-4.1-mini",
        input=input
    )

In [16]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def generateUserInput(jobDescription):
    return f"""
        Here is the job description. Scan and give me result.

        {jobDescription}
    """
    

In [17]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
system_prompt = """
    You are an ATS system that review candidates' resumes for job description. 
    You must help the candidate to tell what needs to be added in the resume based on the job description
    so that they can pass the threshold of 70%. 
    Share important information fromt the description like list of required skills. 
    Talk to the point. Response in markdown.
"""

In [18]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def generateInput(jobDescription):
    return [
        { "role": "system", "content": system_prompt},
        { "role": "user", "content": generateUserInput(jobDescription)}
    ]

In [19]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def getResponseFromAI(input):
    response = callLLM(input)
    return response.output_text

In [20]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def getJobDescriptionKeyPoints(jobDescription): 
    input = generateInput(jobDescription)
    return getResponseFromAI(input)

## 4. 职位描述示例

下一个单元格包含 Priority Technology Holdings 的 **高级软件工程师** 职位的真实职位招聘信息。

这是管道将分析的输入。您可以将其替换为任何职位描述，以测试具有不同角色的助理。

In [21]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
jobDescription = """
About the job
Job title: Senior Software Engineer

Reports to: Director, Engineering

Department: Development

Location: Waterloo, Ontario, Canada





About Priority: 

Priority Technology Holdings, Inc. is a leading financial technology company on a mission to deliver a personalized, easy-to-adopt financial toolset that accelerates cash flow and optimizes working capital for businesses. Our vision is to eliminate the barriers to unlocking revenue - empowering businesses to grow faster and operate smarter.



We achieve this through the Priority Commerce Engine, an innovative platform that combines payables, acquiring, and banking and treasury solutions. This unified approach allows businesses to streamline financial operations, reduce unnecessary costs, and uncover new revenue opportunities.



At Priority, we're driven by results. We expect our people to be known for results - bringing expertise, momentum, and relentless focus to every challenge, helping our clients and each other thrive.





About the Role:

This role is focused on designing, building, and scaling high-impact solutions that improve how we develop, operate, and optimize our platform. The Senior Software Engineer will lead technical design and implementation for complex initiatives, translating ambiguous problems into scalable technical solutions and helping move ideas from concept to production. This role requires strong technical judgment, architectural thinking, and the ability to balance rapid experimentation with production rigor, reliability, and maintainability.



This role is focused on designing, building, and scaling high-impact solutions across three core areas: customer-facing AI innovation, AI-driven risk and fraud, and AI-enabled operational efficiency.



You will lead the development of AI-powered product capabilities, helping prototype and productionize features that deliver meaningful value to customers. This includes working through ambiguity to translate new ideas into reliable, production-ready systems. You will also contribute to building and improving AI-driven risk and fraud systems, ensuring solutions are accurate, scalable, and responsive to evolving threats. In parallel, you will help develop AI-enabled internal tools and workflows that improve how we build and operate the platform, including developer experience, product development processes, and broader business functions.





Responsibilities: 

Owns the design, delivery, and operation of complex services or systems, ensuring high standards for reliability, scalability, and maintainability.
Leads technical design for features and systems, making sound architectural decisions and evaluating trade-offs across performance, reliability, and development velocity.
Drives implementation of solutions that align with system architecture, product requirements, and long-term platform evolution.
Designs and maintains comprehensive automated testing strategies that validate system behavior, prevent regressions, and support safe, rapid delivery.
Ensures systems integrate effectively into CI/CD pipelines with strong quality gates and reliable deployment practices.
Establishes and reinforces engineering standards through code reviews, design reviews, and mentorship.
Owns system reliability by defining and improving observability through metrics, logging, and tracing.
Designs and implements alerting strategies that detect issues early and align with system SLIs/SLOs.
Leads debugging and resolution of complex production issues, driving root cause analysis and long-term fixes.
Defines and tracks system-level KPIs (e.g., latency, error rates, throughput) and contributes to defining product KPIs that measure customer outcomes.
Ensures delivered solutions are measured, observable, and aligned with intended product and business impact.
Collaborates closely with software engineers, product managers, and product designers to shape solutions that deliver reliable and intuitive customer experiences.
Translates product requirements and user workflows into scalable technical designs that maintain system integrity and performance.
Identifies and addresses technical risks, scalability challenges, and reliability gaps across systems.
Drives improvements to system design, development workflows, and engineering practices within the team.
Leverages AI-assisted engineering tools to accelerate development, testing, debugging, and operational analysis while ensuring correctness and maintainability.






What Success Looks Like:

Owns systems that operate reliably, scale effectively, and meet defined performance and availability expectations.
Designs and delivers solutions that balance speed, quality, and long-term maintainability.
Technical decisions improve system architecture, reduce risk, and enable future product evolution.
Systems are observable, measurable, and supported by effective alerting and operational practices.
Production issues are identified, diagnosed, and resolved quickly, with durable fixes that reduce recurrence.
Testing strategies and validation systems provide strong confidence in system behavior and release quality.
System and product KPIs are clearly defined, measured, and used to evaluate success and guide improvements.
Engineering work is consistently aligned with customer outcomes and business impact.
Collaborates effectively with engineers, product managers, and product designers to deliver high-quality product experiences.
Provides technical leadership within the team and is a trusted partner in design and execution decisions.
Elevates team performance through mentorship, code reviews, and improved engineering practices.
Demonstrates strong ownership, accountability, and continuous improvement mindset.




Candidate Requirements:

Required:

5+ years of software engineering experience, with a track record of delivering high-quality software in production environments.
Proven ability to design, build, and maintain reliable services or systems with moderate complexity.
Strong experience contributing to technical design and making sound engineering decisions within a team.
Strong understanding of modern software development practices, including test-driven development (TDD), and building scalable, maintainable, and observable systems.
Experience designing and implementing APIs and services with clear, well-defined contracts.
Solid understanding of web application architecture, including RESTful APIs, backend services, and service-oriented design principles.
Strong understanding of data modeling and data access patterns, including writing efficient queries and contributing to well-structured relational schemas.
Experience working effectively within modern development environments, including version control systems (e.g., Git) and Agile development practices.
Ability to debug complex issues across services using logs, metrics, and traces.
Demonstrated ability to identify technical risks, evaluate trade-offs, and propose effective solutions.
Experience mentoring junior engineers and contributing to team engineering standards and practices.
Experience using AI-assisted development tools to improve productivity, testing, and debugging workflows.
Exposure to integrating AI-enabled features or workflows into applications is a plus.
The primary technology stack for the Innovation Studio team includes: Node.js/NestJS, AWS, Kubernetes, and modern CI/CD and observability tooling, along with AI platforms such as OpenAI, Anthropic, and Gemini. Candidates should have strong proficiency in backend and cloud-native development, with the ability to independently design and deliver complex solutions in distributed systems environments.


Preferred:

Experience in fintech, payments, lending, or other regulated financial systems.
Bachelor’s degree in Computer Science, Engineering, or a related technical field, or equivalent practical experience.
Advanced degree (e.g., Master’s) in a relevant technical discipline.




Work Environment & Culture:

We believe that performance and experience go hand in hand - an exceptional employee experience is earned through contribution. We are a results-driven team, grounded in our core values: ownership, authenticity, service, trust, innovation, and camaraderie.



Our culture is built for those who want to make an impact. We challenge each other to grow, celebrate progress, and support one another through shared goals and real connection. Whether you're building technology, serving clients, or supporting internal teams, you’ll be part of a company that empowers you to perform at your best and be known for results.





Compensation & Benefits:



Compensation range: $113k - $139k CAD

We invest in the whole employee - personally and professionally. Our benefits package is designed to support your well-being, growth, and success - both inside and outside of work.



Financial Wellness

Bonus programs
Financial wellness resources and employee discount programs


Health & Well-being

Medical, dental, and vision coverage
Mental health support for employees and dependents through Lyra Health
Family planning and women’s health benefits through Carrot
Gym membership reimbursement and virtual wellness programs (including yoga)


Time Off

3 weeks PTO to start, with unlimited PTO after year one


Growth & Development

Education expense reimbursement
Leadership development programs
Certified Payments Professional (CPP) certification support


We believe great performance starts with feeling supported - and we’ve built our benefits with that in mind.

 



Traditional Physical Requirements:

Requires prolonged sitting, standing, bending, stooping and stretching.
Requires the ability to lift 10 pounds.
Requires eye-hand coordination, manual dexterity and a normal range of hearing and vision (with or without correction).
 



Join our team at Priority Technology Holdings, Inc. and be part of a dynamic and innovative company that is transforming the financial technology landscape. Together, we can shape the future of payments and banking solutions while providing unmatched value to our clients.


Requirements added by the job poster

• 1+ years of work experience with Acceptance Test‚ÄìDriven Development (ATDD)

• 5+ years of work experience with RESTful WebServices

• 6+ years of work experience with Node.js
"""

## 5. 运行分析

下面的单元格使用示例作业描述调用“getJobDescriptionKeyPoints()”并打印模型的降价响应。

预期产出包括：
- **所需技能和经验**的总结
- **优先**资格
- 关于添加到简历中的内容的可行建议

In [22]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
output = getJobDescriptionKeyPoints(jobDescription)
print(output)

```markdown
# Job Description Summary for Senior Software Engineer at Priority Technology Holdings, Inc.

## Required Skills & Experience:
- 5+ years in software engineering with production delivery.
- 6+ years experience with **Node.js**.
- 5+ years working with **RESTful Web Services**.
- 1+ year experience with **Acceptance Test–Driven Development (ATDD)**.
- Strong in designing, building, and maintaining reliable services/systems of moderate complexity.
- Architectural design skills with sound engineering decisions.
- Test-driven development (TDD) and automated testing strategies experience.
- Experience in API and service design with clear contracts.
- Understanding of web app architecture including RESTful APIs and service-oriented design.
- Strong data modeling and efficient querying skills.
- Familiarity with Git, Agile practices, CI/CD pipelines, metrics, logging, and tracing for observability.
- Debugging complex multi-service issues using logs, metrics, and traces.
- Mentori

## 后续步骤

尝试自己扩展此笔记本：

1. **交换职位描述** — 粘贴您实际申请的职位
2. **添加您的简历** — 将简历和职位描述传递给模型以进行并排差距分析
3. **调整系统提示** — 调整 70% 阈值、音调或输出格式
4. **添加错误处理** — 在调用 API 之前检查是否设置了“OPENAI_API_KEY”

---

*作为法学硕士工程课程的一部分——第一天任务。*